# Figure 5 — three ways to turn a posterior into a decision

The anchor figure of the lecture. One posterior on top; PI, EI and UCB below on a shared x-axis, each with its argmax dropped as a line. Also exports three cumulative versions for a click-build in the deck.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

xs = np.linspace(0, 10, 700)
OX = np.array([1.15, 2.90, 4.30, 6.10, 9.30])
OY = land.f1d(OX)
g = gpmod.GP(gpmod.matern52, ls=0.85, sf=1.0, sn=0.03).fit(OX[:, None], OY)
mu, sd = g.predict(xs[:, None])
fbest = OY.max()

A = {"PI":  gpmod.pi_acq(mu, sd, fbest),
     "EI":  gpmod.ei(mu, sd, fbest),
     "UCB": gpmod.ucb(mu, sd, 2.0)}
COL = {"PI": style.GRAY, "EI": style.RED, "UCB": style.BLUE}
NOTE = {"PI": "chance of beating the best — ignores by how much",
        "EI": "expected size of the improvement",
        "UCB": r"$\mu + 2\sigma$ — the plausible best case"}


def draw(which, fname):
    n = 1 + len(which)
    fig, axes = plt.subplots(n, 1, figsize=(style.FIG_W_FULL, 1.35 + 1.15 * n),
                             sharex=True, gridspec_kw=dict(hspace=0.20,
                             height_ratios=[2.1] + [1.0] * len(which)))
    ax = axes[0]
    ax.plot(xs, land.f1d(xs), "--", color=style.INK, lw=1.1, alpha=0.55,
            label="true objective (never observed)")
    ax.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL, alpha=0.16,
                    lw=0, label=r"posterior $\pm 2\sigma$")
    ax.plot(xs, mu, color=style.TEAL, lw=2.0, label="posterior mean")
    ax.plot(OX, OY, "o", ms=7, color=style.RED, mec="white", mew=1.1,
            label="experiments so far", zorder=6)
    ax.axhline(fbest, color=style.RED, lw=0.8, ls=":", alpha=0.8)
    ax.text(9.95, fbest + 0.05, r"$f^+$", color=style.RED, ha="right", fontsize=11)
    ax.set_ylabel("yield (arb.)")
    ax.legend(loc="lower center", ncol=4, fontsize=9.6, framealpha=0.92,
              facecolor="white")
    ax.set_ylim(-1.9, 2.5)

    for ax, name in zip(axes[1:], which):
        a = A[name]
        ax.plot(xs, a, color=COL[name], lw=1.8)
        ax.fill_between(xs, 0 if name != "UCB" else a.min(), a,
                        color=COL[name], alpha=0.13, lw=0)
        xstar = xs[int(np.argmax(a))]
        ax.axvline(xstar, color=COL[name], lw=1.0, ls="--")
        axes[0].axvline(xstar, color=COL[name], lw=1.0, ls="--", alpha=0.75)
        ax.plot([xstar], [a.max()], "v", ms=8, color=COL[name])
        ax.set_ylabel(name, color=COL[name], fontweight="bold")
        ax.set_yticks([])
        ax.text(0.988, 0.90, NOTE[name], transform=ax.transAxes, va="top",
                ha="right", fontsize=10.6, color=style.GRAY, bbox=dict(fc="white", alpha=0.85, ec="none", pad=2.0))
    axes[-1].set_xlabel("reaction parameter  x")
    style.save(fig, fname, OUT)
    plt.close(fig)


draw(["PI"], "fig_05a_acquisition_pi")
draw(["PI", "EI"], "fig_05b_acquisition_pi_ei")
draw(["PI", "EI", "UCB"], "fig_05_acquisition_panel")

for name, a in A.items():
    print(f"{name:4s} argmax at x = {xs[int(np.argmax(a))]:.2f}")
print("argmax of the posterior mean at x =", round(float(xs[int(np.argmax(mu))]), 2))